<a href="https://colab.research.google.com/github/mohanraju2002/text_to_image/blob/main/text_to__video2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Install dependencies (update torch to latest compatible, use git for diffusers)
!pip install -U torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install git+https://github.com/huggingface/diffusers transformers accelerate imageio[ffmpeg] matplotlib

In [ ]:


import torch
from diffusers import DiffusionPipeline, DPMSolverMultistepScheduler
from diffusers.utils import export_to_video
from IPython.display import HTML
from base64 import b64encode

# Load pipeline with fp16 variant for efficiency
pipe = DiffusionPipeline.from_pretrained(
    "damo-vilab/text-to-video-ms-1.7b",
    torch_dtype=torch.float16,
    variant="fp16"
)
pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config)
pipe.enable_model_cpu_offload()
pipe.enable_vae_slicing()

In [ ]:


# Generate video
prompt = "tiger walking on a beach"
negative_prompt = "low quality, blurry"
num_frames = 50  # ~3 seconds at 10 fps
video_frames = pipe(
    prompt,
    negative_prompt=negative_prompt,
    num_inference_steps=25,
    num_frames=num_frames
).frames[0]  # Take first batch

# Export to MP4
video_path = export_to_video(video_frames, fps=10)
print(f"Video saved to: {video_path}")

# Display video in Colab
def show_video(video_path, video_width=600):
    video_file = open(video_path, "rb").read()
    data_url = "data:video/mp4;base64," + b64encode(video_file).decode()
    return HTML(f"""
    <video width={video_width} controls>
      <source src="{data_url}" type="video/mp4">
    </video>
    """)

show_video(video_path)


  0%|          | 0/25 [00:00<?, ?it/s]

Video saved to: /tmp/tmp1bgovijb.mp4
